# Text Chunking Strategies

How you split documents is one of the most consequential decisions in a RAG pipeline — bad chunks put irrelevant context in the prompt and produce wrong answers.

**The classic failure:** a doc mixing medical research and software engineering. Ask *"How many bugs did engineers fix this year?"* and a careless chunker might return the **medical** section because it contains the word "bug" (as in an insect / pathogen) in an unrelated context.

Four strategies, roughly increasing in sophistication:

| Strategy | Idea | Best when |
|----------|------|-----------|
| **Size-based** | fixed-length character windows (+ overlap) | any document; reliable fallback (incl. code) |
| **Sentence-based** | split into sentences, group N per chunk (+ overlap) | most plain text; good middle ground |
| **Structure-based** | split on headers/sections | you control the format (Markdown, internal reports) |
| **Semantic** | group sentences by meaning (NLP) | best chunks, but expensive/complex |

> Pure-Python lesson — no API calls. Uses the sample `report.md` in this folder.

## Size-based chunking (with overlap)

Divide into equal-length strings. Simple and universal, but it cuts words mid-sentence and can orphan headers from their content. **Overlap** — repeating a few characters from the neighbor — softens the boundaries.

In [1]:
def chunk_by_char(text, chunk_size=150, chunk_overlap=20):
    chunks = []
    start_idx = 0

    while start_idx < len(text):
        end_idx = min(start_idx + chunk_size, len(text))

        chunk_text = text[start_idx:end_idx]
        chunks.append(chunk_text)

        start_idx = (
            end_idx - chunk_overlap if end_idx < len(text) else len(text)
        )

    return chunks

## Sentence-based chunking

Split on sentence boundaries, then group `max_sentences_per_chunk` at a time with `overlap_sentences` of carry-over. Keeps sentences intact — a practical middle ground.

In [2]:
import re


def chunk_by_sentence(text, max_sentences_per_chunk=5, overlap_sentences=1):
    sentences = re.split(r"(?<=[.!?])\s+", text)

    chunks = []
    start_idx = 0

    while start_idx < len(sentences):
        end_idx = min(start_idx + max_sentences_per_chunk, len(sentences))

        current_chunk = sentences[start_idx:end_idx]
        chunks.append(" ".join(current_chunk))

        start_idx += max_sentences_per_chunk - overlap_sentences

        if start_idx < 0:
            start_idx = 0

    return chunks

## Structure-based chunking

Split on the document's own structure. For Markdown, split on the `## ` header marker — each chunk becomes one clean, self-contained section. Cleanest results, but only works when the format is guaranteed.

In [3]:
def chunk_by_section(document_text):
    pattern = r"\n## "
    return re.split(pattern, document_text)

## Semantic chunking (concept only)

The most sophisticated approach: split into sentences, use NLP to measure how related consecutive sentences are, and start a new chunk when the topic shifts. Produces the most coherent chunks but is computationally expensive and more complex to implement — so the course describes it rather than coding it here. (In practice you'd embed each sentence — see the next lesson — and cut where adjacent-sentence similarity drops.)

## Load the sample document

In [4]:
import os
from dotenv import find_dotenv

SECTION = os.path.join(os.path.dirname(find_dotenv()), "building-with-the-claude-api", "05-rag")
REPORT = os.path.join(SECTION, "report.md")

with open(REPORT, "r", encoding="utf-8") as f:
    text = f.read()

print(f"{len(text)} characters loaded from report.md")

18305 characters loaded from report.md


## Compare the strategies

Each produces a different number and shape of chunks. Peek at the count and the first chunk of each.

In [5]:
char_chunks = chunk_by_char(text)
sentence_chunks = chunk_by_sentence(text)
section_chunks = chunk_by_section(text)

print(f"size-based:      {len(char_chunks)} chunks (~150 chars each)")
print(f"sentence-based:  {len(sentence_chunks)} chunks (~5 sentences each)")
print(f"structure-based: {len(section_chunks)} chunks (one per section)")

size-based:      141 chunks (~150 chars each)
sentence-based:  33 chunks (~5 sentences each)
structure-based: 15 chunks (one per section)


In [7]:
# Structure-based gives the cleanest, most meaningful chunks on this well-formatted report.
# Chunk 4 is the whole "Medical Research" section, self-contained:
print(section_chunks[3][:600], "...")

Methodology

The insights compiled within this Annual Interdisciplinary Research Review represent a synthesis of findings drawn from standard departmental reporting cycles, specialized project updates, and cross-functional review meetings conducted throughout the year. Data sources included internal project databases, laboratory notebooks, financial reporting systems, legal case summaries, security incident logs, and minutes from dedicated working groups. A central review committee, comprising representatives nominated by each division head, was tasked with identifying key developments and pot ...


In [8]:
# Size-based, by contrast, slices mid-word regardless of meaning.
# Note the overlap: the tail of one chunk reappears at the head of the next.
for c in char_chunks[:3]:
    print(repr(c), "\n----")

'# **Annual Interdisciplinary Research Review: Cross-Domain Insights**\n\n## Executive Summary\n\nThis report synthesizes the key findings and ongoing rese' 
----
"ngs and ongoing research efforts across the organization's diverse operational and R&D departments for the past fiscal year. Our strength lies in the " 
----
'trength lies in the cross-pollination of ideas and methodologies, driving innovation and addressing complex challenges that transcend traditional disc' 
----


## Choosing a strategy

- **Structure-based** — best results when you control the format (internal reports, Markdown docs like this one).
- **Sentence-based** — solid default for arbitrary prose.
- **Size-based (+ overlap)** — the production go-to: dead simple, never breaks, works on *any* content including code. Not perfect, but consistently reasonable.

There's no single best strategy — it's a trade-off between implementation complexity and chunk quality for *your* documents.

### Aside — chunking a codebase

For code, the meaningful unit isn't a header or a fixed window — it's a **function/class/file**. A structure-based splitter keyed to the language's syntax (via a parser/AST or something like tree-sitter) keeps each chunk a coherent, self-contained definition, so retrieval returns whole functions rather than half a loop body. Size-based (+ overlap) is still the reliable fallback when you can't parse a file. Either way, prepend a little metadata (file path, symbol name) to each chunk so retrieved code carries its location.

Next: **Text embeddings** — turning chunks into vectors so we can retrieve by *meaning*, not just keywords.